# 12F-05 — WLTP/JRC non-BEV EDA

**Question:** What engineering patterns appear in the conventional population?

In [ ]:
from pathlib import Path
import pandas as pd

search_roots = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(path for path in search_roots if (path / "AGENTS.md").exists() and (path / "etl").exists())
JRC_FILE = REPO_ROOT / "etl/data/raw/wltp_jrc/Data_PV_fleet_2021_EU_PYCSIS.xlsx"
jrc = pd.read_excel(JRC_FILE, sheet_name="Sheet1", engine="openpyxl")
assert pd.api.types.is_bool_dtype(jrc["is_electric"]), jrc["is_electric"].dtype
nonbev = jrc[~jrc["is_electric"].fillna(False).astype(bool)].copy()
print(f"{len(nonbev)} non-BEV source records")


## Numeric engineering view

In [ ]:
fields = ["Vehicle mass (WLTP) [kg]", "wltp|f0 [N]", "wltp|f1 [N/(km/h)]", "wltp|f2 [N/(km/h)2]",
          "Declared average CO2 emissions value (OEM) [g/km]"]
numeric = nonbev[fields].apply(pd.to_numeric, errors="coerce")
numeric.describe().T


In [ ]:
nonbev["Fuel type"].value_counts(dropna=False)


## Mass, roadload, and declared CO2

In [ ]:
import matplotlib.pyplot as plt

plot_data = numeric.dropna(subset=["Vehicle mass (WLTP) [kg]", "Declared average CO2 emissions value (OEM) [g/km]"])
ax = plot_data.plot.scatter(x="Vehicle mass (WLTP) [kg]", y="Declared average CO2 emissions value (OEM) [g/km]", alpha=0.55, figsize=(7, 4))
ax.set(title="JRC non-BEV source records")
plt.show()


In [ ]:
numeric.corr(numeric_only=True)["Declared average CO2 emissions value (OEM) [g/km]"].sort_values(ascending=False)


## What did we learn?

This small technical sample supports engineering exploration, not population claims. Correlation helps choose questions; it does not establish causality or a canonical model.